In [ ]:
import torch
import math
import torch.nn as nn

In [ ]:


class LinearLayer:

    def __init__(self,in_features,out_features):
        
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.bias = torch.zeros(out_features)

        # we can use the requires_grad=True to avoid this ,as the torch usually assign the grad=0 for the fresh tensors,these two lines are optional
        self.weights.grad = None  
        self.bias.grad = None
        self.has_bias = True 
        
        #weights are indeed in the reverse shape here, (out,in), because we did multiply with the x-> (B,in_features) @ weights->(out_features,in_features).T 
        # so we apply the transpose ,so we usually assign the weights in the tranpose order
        # there is no mandatory to take the weights only in this way,its all upto you

    # this is for the 2-dimensional matrix,but starting with this makes to learn easy,as you can see the more dim's implementation in raw_transformer.ipynb file
    def forward(self, x):
        self.x  = x
        return x @ self.weights.T+self.bias

    
    def backward(self,grad_out):

        # store the shape of the x to reshape
        x_shape = self.x.shape

        # flatten the higher dimensions to make the dot product work
        grad_flat = grad_out.flatten(0,-2)
        x_flat = self.x.flatten(0,-2)
        
        # (B,in_features) -> (B,out_features) @ (out_features,in_features)
        grad_inputs = grad_flat @ self.weights

        # (out_features,in_features) -> (out_features,B) @ (B,in_features)
        self.weights.grad = grad_flat.T @ x_flat

        # bias shape -> (out_features),derivating it gives us 1 [grad_out*1 + grad_out*1 + grad_out*1 + .........](to the end of how many out_features are..) so we sum them up
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        # Reshape back to the original input dimensions to send back to the prev layers
        return grad_inputs.reshape(x_shape)

B = 2
in_features= 32
out_features = 16

linear = LinearLayer(in_features,out_features)

x = torch.randn(B,in_features)
y = linear.forward(x)
y.shape



# Linear Layer

---

### Why use `nn.Parameter`?

there are many reasons for using the `nn.Parameter`s

1. we can't be able to use `model.parameters()` when we try to use backpropagation in the optimizer
2. we can't move our model weights to `cuda` unless explicitly written
3. it won't save the model weights with `model.state_dict()`

But honestly our main aim is not to oppose torch functions and modules

we need to use them but we need to write the code that can be understood in a way we can upgrade ourself from the custom implementations to torch upgrades

---

### The Handwritten Digit Example

lets consider a famous example of the handwritten digit identification process

there is a image of a digit is given and our role is to find which digit it is by our computer

in this scenario the example considers the gray scale image

now consider a `(30px, 30px)` image so these are the length and breadth — that gives us `30 × 30 = 900` pixel boxes

now there is a image we can see in that `900` boxes — the computer sees each box as a continuous value between `0` and `1` (grayscale intensity — `0` = completely black, `1` = completely white, anything in between is a shade of gray)

how can it distinguish that image, its simple

on which boxes the digit was drawn, because the entire digit itself can't be in a single box right, so it would be distributed across the different boxes

now you can check the corners — there is no digit drawn in those edge boxes so that pixel value is close to `0` (very dark/empty)

and boxes where the digit was actually drawn are closer to `1` depending on the intensity of the ink in that pixel

so these are the neurons — they are basically the pixel intensity values between `0` and `1`

---

### Weights, Inputs, Outputs

and now we need to come up with the solution of the weights

now before coming to the weights we need to know about the two key parameters:

- `in_features = 900` — these are the 900 neurons (one per pixel)
- `out_features = 10` — we only need 10, why? because in this case, our goal is to reduce the 900 neurons into the 10 neurons which say the digit is either `0, 1 .... 9`, any one of them

---

### What is the bias?

`bias` -> this is the threshold required to give a small push to the neuron, because sometimes they need to fire up in a way that, the answer might be the digit `8`

but the neuron that is a bit low on the activation energy to trigger, so we apply this bias to use as a threshold and it will trigger the digit `8` as the answer from the model

- **Positive bias:** Lowers the barrier so the neuron fires even with weak input.
- **Negative bias:** Raises the barrier so the neuron stays quiet unless the evidence is overwhelming.

Even when inputs are dead silent, the bias represents the baseline prior probability or default state of the neuron.

---

### What exactly are the weights?

the interesting part is the weights

so the weights are the matrix which are in the shape of the `(in_features, out_features)` or `(out_features, in_features)` -> depends on your wish as you can transpose at the forward method

input neurons (`in_features`) are gonna effect the output neurons (`out_features`) because we are reducing them to the size of the output neurons

so, each input neuron is connected to the each output neuron

(but why? the reason is, because of that we can know how far does it effects the other numbers as the output, as the input neuron 1 might contribute more to the digit `1` and very less to the digit `9`

in this case, these all 900 neurons will be taken into the charge and the final neuron is activated, it is that digit)

so as you can see, we can say:

> Each of the 900 input neurons connects to ALL 10 output neurons.
> The weight matrix `(10, 900)` stores these connections: row `i` contains 900 weights showing how much EACH input pixel influences output neuron `i` (which represents digit `i`)

---

### Without `nn.Parameter`

what happens when you don't use the `nn.Parameter`

then you need to manually track gradients yourself:

```python
self.weights_grad = None
self.bias_grad    = None
```

you also have to set `requires_grad=True` on each tensor manually so that torch actually calculates the gradients for you via autograd

and to read the gradient later you'd do `tensor.grad` — but only on tensors that had `requires_grad=True`

---

### With `nn.Parameter` (the clean way)

you can use the `nn.Parameter` if you want, after inheriting from the `nn.Module` — it does all of the above automatically

**Benefits of `nn.Parameter`:**

1. automatic parameter registration — no manual tracking needed
2. no need of writing `requires_grad=True` — it's on by default
3. device migration (`cuda`/`cpu`) — `model.to(device)` moves everything
4. saving the states of the model — `model.state_dict()` includes it automatically

when you create the `torch.randn()` pytorch creates the data attribute related to that

so we can use the `p.data` to access or modify the raw tensor values directly

---

### Kaiming Scaling

using the `torch.randn()` method we can initialize:
- mean -> `0`
- variance -> `1`

and with the kaiming we can initialize:
- mean -> `0`
- var -> `(std)^2`

---

### Why does the mean stay 0?

imagine with no bias

`y = xw`

if we did imagine the `x` and `w` are the independent random variables (as the data doesn't know how we initialized the weights)

The Expected Value (Mean) of a product of two independent variables is:

`E[Y] = E[X·W] = E[X]·E[W]`

as the `torch.randn` makes the mean `0` to make them symmetrical around zero, `E[W] = 0`

and `E[Y] = 0`

Because the mean of `Y` is always `0`, the mean of the gradient `E[∂L/∂W]` also remains `0` throughout training (assuming no bias).

This is great because it means your updates `ΔW` are centered around zero, preventing all your weights from drifting permanently positive or negative.

If `E[W] ≠ 0`, your updates would have a systematic bias, causing a "drift" effect where all neurons learn the same thing.

---

### Why does variance matter?

`Y = X1·W1 + X2·W2 + X3·W3 + ....`

`Var(Y) = sum of the individual variances`

`Var(Xi·Wi) = Var(Xi) · Var(Wi)`

so if we did assume all the inputs and weights have the same variance:

`Var(Y) = Σᵢ (σX² · σW²) = D · σX² · σW²`

`Var(Y) = D · Var(X) · Var(W)`

| Init method | `Var(W)` | Result |
|---|---|---|
| `torch.randn` (Std = 1) | `1` | `Var(Y) = D · Var(X)` — blows up! |
| Kaiming (Std = `√(2/D)`) | `2/D` | `Var(Y) = 2 · Var(X)` — stable ✅ |